# Counterfactual Evaluation Pipeline  
## Surrogate Consistency and PatchCore Validation


### Configuration

In [9]:
# ============================================================
# Imports
# ============================================================

import json
import numpy as np
import pandas as pd
import time
import zlib
import random
import torch

from pathlib import Path

from catboost import CatBoostRegressor

from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from PIL import Image, ImageFilter

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score


In [10]:
# ============================================================
# Paths
# ============================================================


ART_DIR    = Path("surrogate_pkl_cfs_metadata")
SPLITS_DIR = Path("patchcore_splits")

CFS_PATH   = ART_DIR / "cfs.csv"
META_PATH  = ART_DIR / "metadata.json"
MODEL_PATH = ART_DIR / "surrogate_catboost.cbm"

TRAIN_PATH = SPLITS_DIR / "df_train_good.csv"
VAL_PATH   = SPLITS_DIR / "df_val.csv"
TEST_PATH  = SPLITS_DIR / "df_final.csv"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


assert CFS_PATH.is_file(), CFS_PATH
assert META_PATH.is_file(), META_PATH
assert MODEL_PATH.is_file(), MODEL_PATH
assert TRAIN_PATH.is_file(), TRAIN_PATH
assert VAL_PATH.is_file(), VAL_PATH
assert TEST_PATH.is_file(), TEST_PATH

device: cuda


In [11]:
# -----------------------------
# 1) Load Metadata and Counterfactuals
# -----------------------------

with open(META_PATH, "r") as f:
    meta = json.load(f)

FEATURES = meta["features"]
CATEGORICAL = meta["categorical"]
TARGET = meta["target"]

cfs = pd.read_csv(CFS_PATH)

for c in CATEGORICAL:
    cfs[c] = cfs[c].astype(str)

print("Counterfactuals loaded:", cfs.shape)
display(cfs.head())


Counterfactuals loaded: (1, 15)


,cf_id,params_backbone,params_batch_size,params_center_crop_key,params_coreset_sampling_ratio,params_image_size_key,params_layers_key,params_max_patches_per_image,params_num_neighbors,params_pre_trained,params_reduction,params_soft_corruption_level,params_soft_review_budget,params_soft_train_fraction,value
0,0,resnet50,8,0.875,0.099,256,l3,1024,2,True,mean,none,0.365,0.89,0.911063


### Step 1: Surrogate Consistency

In [12]:
# -----------------------------
# Step 1 — Surrogate prediction gap
# -----------------------------

cb = CatBoostRegressor()
cb.load_model(str(MODEL_PATH))

X = cfs[FEATURES].copy()

cfs["surrogate_pred"] = cb.predict(X).astype(float)
cfs["gap_pred_minus_cfvalue"] = cfs["surrogate_pred"] - cfs[TARGET]

display(cfs[["cf_id", TARGET, "surrogate_pred", "gap_pred_minus_cfvalue"]])
print("\nGap stats:")
print(cfs["gap_pred_minus_cfvalue"].describe())


,cf_id,value,surrogate_pred,gap_pred_minus_cfvalue
0,0,0.911063,0.911063,-1.294360e-08



Gap stats:
count    1.000000e+00
mean    -1.294360e-08
std               NaN
min     -1.294360e-08
25%     -1.294360e-08
50%     -1.294360e-08
75%     -1.294360e-08
max     -1.294360e-08
Name: gap_pred_minus_cfvalue, dtype: float64


### Step 2: Main Model Validation

In [13]:
# ============================================================
# Load splits
# ============================================================

df_train_good = pd.read_csv(TRAIN_PATH)
df_val = pd.read_csv(VAL_PATH)
df_final = pd.read_csv(TEST_PATH)

print("train_good:", df_train_good.shape)
print("val:", df_val.shape)
print("test:", df_final.shape)

train_good: (4538, 2)
val: (867, 2)
test: (867, 2)


In [14]:
# ============================================================
# PatchCore utilities (as used in notebook 1)
# ============================================================

def _stable_int_seed(text: str, seed: int) -> int:
    return (zlib.crc32(text.encode("utf-8")) + seed) & 0xFFFFFFFF


def corrupt_pil(img: Image.Image, level: str, seed_int: int) -> Image.Image:
    if level == "none":
        return img

    rng = np.random.default_rng(seed_int)
    out = img

    if level in {"mild", "strong"}:
        radius = 0.6 if level == "mild" else 1.2
        out = out.filter(ImageFilter.GaussianBlur(radius=radius))

    if level == "strong":
        arr = np.asarray(out).astype(np.float32)
        noise_std = 8.0
        noise = rng.normal(0.0, noise_std, size=arr.shape).astype(np.float32)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        out = Image.fromarray(arr)

    return out


class ImagePathDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preproc: transforms.Compose,
                 corruption_level: str = "none", apply_corruption: bool = False):
        self.paths = df["path"].tolist()
        self.labels = df["label"].to_numpy().astype(int)
        self.preproc = preproc
        self.corruption_level = corruption_level
        self.apply_corruption = apply_corruption

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")

        if self.apply_corruption and self.corruption_level != "none":
            s = _stable_int_seed(path, SEED)
            img = corrupt_pil(img, self.corruption_level, s)

        x = self.preproc(img)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def make_preproc(image_size: tuple[int, int], center_crop_size: tuple[int, int] | None):
    t = [transforms.Resize(image_size)]
    if center_crop_size is not None:
        t.append(transforms.CenterCrop(center_crop_size))
    t += [
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406),
                             std=(0.229, 0.224, 0.225)),
    ]
    return transforms.Compose(t)


def make_loaders(df_train_good_sub: pd.DataFrame,
                 image_size: tuple[int, int],
                 center_crop_size: tuple[int, int] | None,
                 batch_size: int,
                 corruption_level: str):
    preproc = make_preproc(image_size=image_size, center_crop_size=center_crop_size)

    train_ds = ImagePathDataset(df_train_good_sub, preproc, corruption_level=corruption_level, apply_corruption=True)
    val_ds   = ImagePathDataset(df_val,          preproc, corruption_level="none", apply_corruption=False)
    final_ds = ImagePathDataset(df_final,        preproc, corruption_level="none", apply_corruption=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    final_loader = DataLoader(final_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, final_loader


def build_backbone(backbone: str, pre_trained: bool) -> torch.nn.Module:
    if backbone == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pre_trained else None)
    elif backbone == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT if pre_trained else None)
    elif backbone == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pre_trained else None)
    elif backbone == "wide_resnet50_2":
        m = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.DEFAULT if pre_trained else None)
    else:
        raise ValueError(f"Unsupported backbone: {backbone}")

    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    return m


class FeatureExtractor(torch.nn.Module):
    def __init__(self, backbone: str, layers: tuple[str, ...], pre_trained: bool):
        super().__init__()
        self.backbone = build_backbone(backbone, pre_trained)
        self.layers = layers
        self._features: dict[str, torch.Tensor] = {}

        named = dict(self.backbone.named_modules())
        for name in layers:
            if name not in named:
                raise ValueError(f"Layer '{name}' not found in backbone '{backbone}'")
            named[name].register_forward_hook(self._hook(name))

    def _hook(self, name: str):
        def fn(module, inp, out):
            self._features[name] = out
        return fn

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        _ = self.backbone(x)
        return [self._features[name] for name in self.layers]


def to_patch_embeddings(feat_maps: list[torch.Tensor], max_patches_per_image: int | None = None, seed: int = 0) -> torch.Tensor:
    H = max(f.shape[-2] for f in feat_maps)
    W = max(f.shape[-1] for f in feat_maps)

    resized = []
    for f in feat_maps:
        if (f.shape[-2], f.shape[-1]) != (H, W):
            f = torch.nn.functional.interpolate(f, size=(H, W), mode="bilinear", align_corners=False)
        resized.append(f)

    f_cat = torch.cat(resized, dim=1)  # [B, D, H, W]
    B, D, _, _ = f_cat.shape

    f_flat = f_cat.flatten(2).transpose(1, 2).contiguous()  # [B, P, D]
    P = f_flat.shape[1]

    if max_patches_per_image is not None and max_patches_per_image < P:
        g = torch.Generator(device=f_flat.device)
        g.manual_seed(seed)
        idx = torch.randperm(P, generator=g, device=f_flat.device)[:max_patches_per_image]
        f_flat = f_flat[:, idx, :]  # [B, P', D]

    return f_flat.reshape(-1, D)


def coreset_sample(embeddings: np.ndarray, ratio: float, seed: int) -> np.ndarray:
    n = embeddings.shape[0]
    m = max(1, int(n * ratio))
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=m, replace=False)
    return embeddings[idx]


@torch.no_grad()
def fit_memory_bank(extractor: FeatureExtractor,
                    train_loader: DataLoader,
                    coreset_sampling_ratio: float,
                    max_patches_per_image: int) -> np.ndarray:
    extractor = extractor.to(DEVICE).eval()

    kept = []
    for x, _ in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)

        patches = to_patch_embeddings(feats, max_patches_per_image=max_patches_per_image, seed=SEED)
        kept.append(patches.detach().cpu().numpy())

        del x, feats, patches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    emb = np.vstack(kept)
    return coreset_sample(emb, ratio=coreset_sampling_ratio, seed=SEED)


@torch.no_grad()
def score_images(extractor: FeatureExtractor,
                 loader: DataLoader,
                 memory_bank: np.ndarray,
                 num_neighbors: int,
                 reduction: str,
                 max_patches_per_image: int) -> tuple[np.ndarray, np.ndarray]:
    extractor = extractor.to(DEVICE).eval()

    nn = NearestNeighbors(n_neighbors=num_neighbors, algorithm="auto")
    nn.fit(memory_bank)

    y_true_list, y_score_list = [], []

    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)

        patches = to_patch_embeddings(feats, max_patches_per_image=max_patches_per_image, seed=SEED).detach().cpu().numpy()

        dists, _ = nn.kneighbors(patches, return_distance=True)
        patch_scores = dists.mean(axis=1)

        B = y.shape[0]
        P = patch_scores.shape[0] // B
        patch_scores = patch_scores.reshape(B, P)

        img_scores = patch_scores.max(axis=1) if reduction == "max" else patch_scores.mean(axis=1)

        y_true_list.append(y.numpy())
        y_score_list.append(img_scores)

        del x, feats, patches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return np.concatenate(y_true_list), np.concatenate(y_score_list)


def threshold_from_review_budget(y_score: np.ndarray, budget: float) -> float:
    assert 0.0 < budget < 1.0
    n = y_score.shape[0]
    k = max(1, int(np.ceil(budget * n)))
    thr = np.partition(y_score, -k)[-k]
    return float(thr)


In [15]:
# ============================================================
# Configuration decoding + single-run evaluator
# ============================================================

LAYERS_MAP = {
    "l2_l3": ("layer2", "layer3"),
    "l1_l2_l3": ("layer1", "layer2", "layer3"),
    "l3": ("layer3",),
    "l2": ("layer2",),
}

IMAGE_SIZE_MAP = {
    "224": (224, 224),
    "256": (256, 256),
    "320": (320, 320),
}

def _as_bool(x):
    if isinstance(x, bool):
        return x
    s = str(x).strip().lower()
    return s in {"1", "true", "yes", "y", "t"}


def run_patchcore_and_score(cfg: dict, seed_offset: int = 0) -> dict:
    # unpack config (CF columns)
    backbone = str(cfg["params_backbone"])
    batch_size = int(cfg["params_batch_size"])
    center_crop_key = str(cfg["params_center_crop_key"])
    coreset_sampling_ratio = float(cfg["params_coreset_sampling_ratio"])
    image_size_key = str(cfg["params_image_size_key"])
    layers_key = str(cfg["params_layers_key"])
    max_patches_per_image = int(cfg["params_max_patches_per_image"])
    num_neighbors = int(cfg["params_num_neighbors"])
    pre_trained = _as_bool(cfg["params_pre_trained"])
    reduction = str(cfg["params_reduction"])

    soft_corruption_level = str(cfg["params_soft_corruption_level"])
    soft_review_budget = float(cfg["params_soft_review_budget"])
    soft_train_fraction = float(cfg["params_soft_train_fraction"])

    image_size = IMAGE_SIZE_MAP[image_size_key]
    layers = LAYERS_MAP[layers_key]

    if center_crop_key == "none":
        center_crop_size = None
    elif center_crop_key == "0.875":
        s = image_size[0]
        center_crop_size = (int(0.875 * s), int(0.875 * s))
    else:
        raise ValueError(center_crop_key)

    # apply soft_train_fraction (subsample train-good)
    if soft_train_fraction >= 1.0:
        df_sub = df_train_good
    else:
        df_sub = df_train_good.sample(frac=soft_train_fraction, random_state=SEED + seed_offset)

    train_loader, val_loader, test_loader = make_loaders(
        df_train_good_sub=df_sub,
        image_size=image_size,
        center_crop_size=center_crop_size,
        batch_size=batch_size,
        corruption_level=soft_corruption_level,
    )

    t0 = time.time()

    extractor = FeatureExtractor(backbone=backbone, layers=layers, pre_trained=pre_trained)

    memory_bank = fit_memory_bank(
        extractor,
        train_loader,
        coreset_sampling_ratio=coreset_sampling_ratio,
        max_patches_per_image=max_patches_per_image,
    )

    # val: calibrate threshold at budget
    y_true_val, y_score_val = score_images(
        extractor, val_loader, memory_bank,
        num_neighbors=num_neighbors,
        reduction=reduction,
        max_patches_per_image=max_patches_per_image,
    )
    thr = threshold_from_review_budget(y_score_val, soft_review_budget)

    # test: apply calibrated threshold
    y_true_test, y_score_test = score_images(
        extractor, test_loader, memory_bank,
        num_neighbors=num_neighbors,
        reduction=reduction,
        max_patches_per_image=max_patches_per_image,
    )

    y_pred_test = (y_score_test >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true_test, y_pred_test, average="binary", zero_division=0)
    auroc = roc_auc_score(y_true_test, y_score_test)

    runtime_sec = time.time() - t0

    return {
        "threshold_val_budget": float(thr),
        "precision_test": float(p),
        "recall_test": float(r),
        "f1_test": float(f1),
        "auroc_test": float(auroc),
        "runtime_sec": float(runtime_sec),
        "train_good_used": int(len(df_sub)),
        "memory_bank_size": int(memory_bank.shape[0]),
    }


In [16]:
# ============================================================
# Run Step 2 for all counterfactuals + compute gaps
# ============================================================

rows = []

for _, r in cfs.iterrows():
    cfg = r[FEATURES].to_dict()
    out = run_patchcore_and_score(cfg, seed_offset=int(r["cf_id"]))

    cf_value = float(r[TARGET])          # DiCE "value"
    test_f1  = float(out["f1_test"])

    rows.append({
        "cf_id": int(r["cf_id"]),
        "cf_value": cf_value,
        "surrogate_pred": float(r["surrogate_pred"]),
        "gap_step1_pred_minus_cfvalue": float(r["gap_pred_minus_cfvalue"]),
        "f1_test": test_f1,
        "auroc_test": float(out["auroc_test"]),
        "gap_test_minus_cfvalue": test_f1 - cf_value,
        **out,
    })

eval_df = (
    pd.DataFrame(rows)
      .sort_values("f1_test", ascending=False)
      .reset_index(drop=True)
)

display(eval_df[["cf_id", "cf_value", "f1_test", "gap_test_minus_cfvalue", "auroc_test", "runtime_sec"]])

print("\nTest-vs-CFValue gap stats:")
print(eval_df["gap_test_minus_cfvalue"].describe())


,cf_id,cf_value,f1_test,gap_test_minus_cfvalue,auroc_test,runtime_sec
0,0,0.911063,0.878444,-0.032619,0.973874,81.791073



Test-vs-CFValue gap stats:
count    1.000000
mean    -0.032619
std           NaN
min     -0.032619
25%     -0.032619
50%     -0.032619
75%     -0.032619
max     -0.032619
Name: gap_test_minus_cfvalue, dtype: float64
